# TrimCI · `ground_state` Tutorial

TrimCI's friendly entry point: **three parameters, one CI calculation**.

```python
result = trimci.ground_state(problem, n_dets=100_000, effort=1.0)
```

- `problem` — FCIDUMP path, `(h1, eri, n_elec)` / `(h1, eri, n_elec, e_nuc)` tuple, **or** PySCF mean-field object
- `n_dets` — final determinant count; controls accuracy
- `effort` — workload multiplier; `1.0` matches `trimci.TrimCI_skill` production defaults; `0.4` is the measured sweet spot

The returned `Result` has a three-layer structure plus a few top-level shortcuts. This notebook walks through: quickstart → reading results → tuning knobs → passing integrals directly → save/load.

> Full reference in the companion [`ground_state_tutorial.html`](./ground_state_tutorial.html) (decision tree, FAQ, longer exposition).

## 0 · Setup

Verify the environment (should be the `dev_qc` conda env):

In [1]:
import sys, numpy as np
import trimci
from trimci import Config, Result

print(f"python   : {sys.version.split()[0]}")
print(f"trimci   : {trimci.__version__}")
print(f"numpy    : {np.__version__}")

∣NK⟩ Tip: You can use flax.linen, flax.nnx and equinox to define neural networks.

python   : 3.12.12
trimci   : 0.2.0
numpy    : 2.3.5


## 1 · Three-line quickstart (FCIDUMP)


In [2]:
FCIDUMP = "FCIDUMP_H8_r2.40_sto-3g"

result = trimci.ground_state(FCIDUMP, n_dets=100, effort=0.04)
print(result)

namespace(threshold=0.01, initial_pool_size=100, pool_core_ratio=40, pool_build_strategy='heat_bath', max_final_dets=10, max_rounds=4, core_set_ratio=[1, 1.1], first_cycle_keep_size=10, num_groups=20, local_trim_keep_ratio=4, num_runs=3, load_initial_dets=False, verbose=False, initial_dets_dict={'reference': 1, 'random': [1, 10000]}, backend='auto', orbital_optimization=True, loaded_dets_randomness=0.1, optimizer_options_dict={'optimizer': 'auto_bfgs', 'cycles': 5, 'maxiter': 100, 'tracking_dets': True, 'davidson_tol': 1e-07, 'ftol': 1e-08}, n_cpu_workers=1, n_gpu_workers=0, verbosity=1, psym=8, fcidump_path='FCIDUMP_H8_r2.40_sto-3g', nuclear_repulsion=3.030169505387141)
[C++ Workflow] Starting with 4899 initial determinants
[C++ Workflow] Total possible configurations: 4900
[C++ Workflow] Sparsity detected: h1=0.428571, eri=0.500977
[C++ Workflow] Double exc table: 28 entries in 0.002959s
[C++] Iteration 1/200000
[PoolBuild] Starting pool build: target_size=4900, threshold=0.01, mode=


[matvec_dressed profile] N=64, 10 threads, main_loop=0.1 ms
  Connections (upper-tri): total=663
    Ch1 SAME_ALPHA_S: 80 (12.1%)
    Ch1 SAME_ALPHA_D: 76 (11.5%)
    Ch3 SAME_BETA_S:  67 (10.1%)
    Ch3 SAME_BETA_D:  69 (10.4%)
    Ch2 MIXED_D:      371 (56.0%)
  Avg connections/det: 10.4


`print(result)` invokes `__repr__`, a single-line summary of variational energy / PT2 / determinant count. For more detail use `summary()`:

In [3]:
print(result.summary())

TrimCI Calculation Summary
  Energy (var):     -3.7433399864 Ha
  Energy (PT2):     -3.7447564962 Ha  (ΔPT2 = -0.001417)
  Determinants:     100  (Phase 0: 10)
  Wall (Phase 0):   20.6 s
  Wall (Expansion): 0.9 s (13 rounds)
  Wall (total):     21.5 s


## 2 · The three-layer structure of `Result`

| Layer | Contents | Key fields |
|---|---|---|
| `result.energies`     | Energy decomposition (Ha) | `var`, `pt2_correction`, `pt2`, `nuclear`, `history` |
| `result.wavefunction` | \|Ψ⟩ = Σ cᵢ\|αᵢ,βᵢ⟩ | `coefficients`/`coeffs`, `alpha_bits`, `beta_bits`, `dets`, `leading(k)` |
| `result.diagnostics`  | Performance & config | `n_dets_phase0`, `wall_phase0`, `wall_expansion`, `n_fci`, `config` |

Plus a few top-level shortcuts (most commonly used): `result.energy` / `result.energy_pt2` / `result.n_dets` / `result.wall_time`.

In [ ]:
# Top-level shortcuts
print(f"E_var      : {result.energy:.10f} Ha")
print(f"E_var+PT2  : {result.energy_pt2:.10f} Ha" if result.energy_pt2 is not None else "PT2: not computed")
print(f"n_dets     : {result.n_dets}")
print(f"wall_time  : {result.wall_time:.2f} s")

In [ ]:
# Energy decomposition
e = result.energies
print(f"variational E : {e.var:.10f} Ha")
print(f"ΔE_PT2        : {e.pt2_correction:+.3e} Ha" if e.pt2_correction is not None else "no PT2")
print(f"E_nuc         : {e.nuclear:.6f} Ha")
print(f"E_var history : {[f'{x:.6f}' for x in e.history[:5]]} ... (len={len(e.history)})")

### 2.1 Wavefunction — the most significant determinants

In [ ]:
wf = result.wavefunction
print(f"n_dets   : {wf.n_dets}")
print(f"n_orb    : {wf.n_orb}")
print(f"is_complex   : {wf.is_complex}")
print(f"is_normalized: {wf.is_normalized}")
print()
print("Top-5 dets by |c|:")
print(f"  {'α bitstring':<8}  {'β bitstring':<8}  {'coefficient':>14}")
for a, b, c in wf.leading(5):
    print(f"  {a}  {b}  {c:>+14.8f}")

### 2.2 Coefficients / determinant arrays (for downstream computation)

The wavefunction is backed by plain numpy arrays you can take directly:

In [ ]:
coeffs = wf.coeffs              # (N,) float64 / complex128
dets   = wf.dets                # (N, 2K) uint64, [α_segs..., β_segs...]
alpha  = wf.alpha_bits          # (N, K) uint64
beta   = wf.beta_bits           # (N, K) uint64

print(f"coeffs : shape={coeffs.shape}  dtype={coeffs.dtype}")
print(f"dets   : shape={dets.shape}  dtype={dets.dtype}")
print(f"⟨Ψ|Ψ⟩  = {(np.abs(coeffs)**2).sum():.10f}")

In [ ]:
# Per-orbital α/β occupation numbers ⟨n_p⟩
occ = wf.occupation_numbers()   # shape (n_orb, 2)
print(f"orbital    α       β       total")
for p in range(wf.n_orb):
    a, b = occ[p]
    print(f"  {p:3d}    {a:.4f}  {b:.4f}  {a+b:.4f}")

### 2.3 Diagnostics

In [ ]:
d = result.diagnostics
print(f"n_dets_phase0     : {d.n_dets_phase0}")
print(f"n_dets_final      : {d.n_dets_final}")
print(f"wall_phase0       : {d.wall_phase0:.2f} s")
print(f"wall_expansion    : {d.wall_expansion:.2f} s")
print(f"expansion_rounds  : {d.expansion_rounds}")
print(f"n_fci (Hilbert)   : {d.n_fci}    # = C(n_orb,n_α) × C(n_orb,n_β)")
print(f"orbopt_dir        : {d.orbopt_dir}")
print(f"checkpoint_dir    : {d.checkpoint_dir}")

**Automatic `n_fci` clamping**: if `n_dets > n_fci`, TrimCI auto-truncates to `n_fci` to avoid pointless expansion. For example H4's FCI dimension is 36, so requesting `n_dets=10⁸` actually runs only to 36.

## 3 · Tuning knobs: `n_dets` and `effort`

**Empirical findings** (from `(n_dets × effort)` sweep experiments on Fe₄S₄):

1. `n_dets` is the **dominant axis** — for higher accuracy, grow `n_dets` first, not effort
2. `effort=0.4` is already the sweet spot; higher effort shows diminishing returns and can increase run-to-run variance
3. `effort=1.0` = `TrimCI_skill` default (num_runs=64, BFGS 100 iter)
4. `effort=0.05` is for smoke tests

Compare two `n_dets` values below:

In [ ]:
import time

for n in [100, 500]:
    t0 = time.time()
    r = trimci.ground_state(FCIDUMP, n_dets=n, effort=0.4)
    e_pt2_str = f"{r.energy_pt2:.8f}" if r.energy_pt2 is not None else "   N/A    "
    print(f"n_dets={n:>4}  E_var={r.energy:.8f}  E_PT2={e_pt2_str}  "
          f"wall={time.time()-t0:.1f}s")

## 4 · Passing integrals directly: `(h1, eri, n_elec[, e_nuc])`

No FCIDUMP, only integral arrays? Also three lines. The `ms2` kwarg sets spin on the tuple path; FCIDUMP and PySCF mf paths take spin from their own sources (file `MS2` header, `mol.spin`).

In [ ]:
from pyscf import gto, scf, ao2mo

# Build an H4 PySCF mean-field
mol = gto.M(atom='H 0 0 0; H 0 0 1; H 0 0 2; H 0 0 3', basis='sto-3g', verbose=0)
mf = scf.RHF(mol).run()
n_orb = mol.nao
n_elec = mol.nelectron

# AO → MO integral transform
h1 = mf.mo_coeff.T @ mf.get_hcore() @ mf.mo_coeff
eri = ao2mo.full(mol, mf.mo_coeff, compact=False).reshape(n_orb, n_orb, n_orb, n_orb)
e_nuc = mol.energy_nuc()

print(f"n_orb={n_orb}  n_elec={n_elec}  e_nuc={e_nuc:.6f}")

result_int = trimci.ground_state((h1, eri, n_elec, e_nuc), n_dets=200, effort=0.4)
print(result_int)

**Open-shell / non-zero spin** is supported across all input modes:

- **Tuple**: pass `ms2 = 2·S_z` directly (`1` = doublet, `2` = triplet, ...)
- **FCIDUMP**: the file's `MS2` header determines spin
- **PySCF mf**: set `mol.spin` before SCF (use `ROHF` / `UHF`)

```python
# H3 doublet (3 electrons) via tuple path
result = trimci.ground_state((h1, eri, 3), n_dets=50, ms2=1)
```

## 5 · Reproducibility: `Config.seed`

Default `seed=None` → fresh entropy → different each run (preserves legacy behavior).

Set `seed=<int>` → fully reproducible (same seed → bit-identical `Result`). Sub-seeds are derived deterministically for the internal randomness (Phase 0 sampling, PT2 sketch, tracking_dets random walk).

In [ ]:
from trimci.api import _config_from_effort, _split_n_dets

p0, p1, p2 = _split_n_dets(200)
cfg = _config_from_effort(0.4, p0, p1, p2, work_dir=None)
cfg.seed = 42
cfg.verbosity = 0

r1 = trimci.ground_state_from_fcidump(FCIDUMP, config=cfg)
r2 = trimci.ground_state_from_fcidump(FCIDUMP, config=cfg)
print(f"E_var (seed=42 run 1): {r1.energy:.12f}")
print(f"E_var (seed=42 run 2): {r2.energy:.12f}")
print(f"identical? {r1.energy == r2.energy}")

## 6 · Save / load

Recommended: `.npz` (handles large dets arrays well); `.json` is also supported but slow for large wavefunctions.

In [ ]:
result.save("/tmp/h4_result.npz")

r_loaded = Result.load("/tmp/h4_result.npz")
print(f"loaded: {r_loaded}")
print(f"E_var match : {r_loaded.energy == result.energy}")
print(f"n_dets match: {r_loaded.n_dets == result.n_dets}")

## 7 · Final orbitals (lazy load from disk)

`ground_state` enables orbopt by default, so the final basis has been rotated. Three attributes lazy-load from `work_dir` on demand:

In [ ]:
h1_final = result.final_h1          # (n_orb, n_orb)
eri_final = result.final_eri        # (n_orb,)*4
U = result.orbital_rotation         # C_final = C_input @ U

if h1_final is not None:
    print(f"final_h1  : shape={h1_final.shape}")
if eri_final is not None:
    print(f"final_eri : shape={eri_final.shape}")
if U is not None:
    print(f"U_total   : shape={U.shape}  unitary? {np.allclose(U.T @ U, np.eye(U.shape[0]), atol=1e-6)}")

## 8 · Advanced: custom `Config`

`ground_state(problem, n_dets, effort)` is internally a wrapper that builds a `Config` from `n_dets` and `effort`. For fine-grained tuning, build a `Config` yourself and pass it via `config=`:

```python
from trimci import Config

cfg = Config(
    # Phase 0
    num_runs=32, orbopt_cycles=5, max_dets_phase0=200,
    # Phase 1
    max_dets_phase1=10_000, orbopt_max_iter=100,
    # Phase 2
    max_dets_phase2=1_000_000, pt2_correction=True, pt2_only_last=True,
    # I/O
    work_dir="./my_run", verbosity=1,
    # Reproducibility
    seed=42,
)
result = trimci.ground_state(FCIDUMP, config=cfg)
```

`config=` works with any `problem` type (FCIDUMP path, integrals tuple, or PySCF mf). When set, `n_dets` and `effort` are ignored.

Full field list (including the `phase0_overrides` / `phase1_overrides` / `phase2_overrides` escape hatches):

```python
from dataclasses import fields
for f in fields(Config):
    print(f"{f.name:30s}  default={f.default!r}")
```

## 9 · Entry points

| Entry point | When to use |
|---|---|
| `ground_state(problem, n_dets, effort, *, config=, active=, ms2=)` | **unified — accepts FCIDUMP path, integrals tuple, or PySCF mf object** |
| `ground_state_from_fcidump(path, config=..., **overrides)` | shorthand `Config(...)` field overrides via kwargs |
| `ground_state_from_integrals(h1, eri, n_elec, *, ms2=, e_nuc=, config=, **overrides)` | same, for raw integrals |
| `ground_state_from_pyscf(mf, active=, config=, **overrides)` | same, for PySCF mf |

All return the same `Result` type; downstream code is identical. Prefer `ground_state` unless you need the `**config_overrides` keyword shorthand.

---

**Next steps**
- Decision tree / FAQ / longer exposition → [`ground_state_tutorial.html`](./ground_state_tutorial.html)
- Source → [`py/trimci/api.py`](../py/trimci/api.py)